In [ ]:
import random
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import time
import sys

np.random.seed(42)

CARD_VALUES = {
    "2": 2, "3": 3, "4": 4, "5": 5,
    "6": 6, "7": 7, "8": 8, "9": 9,
    "10": 10, "J": 10, "Q": 10, "K": 10, "A": 11
}
SUITS = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
DECK = [f"{value} of {suit}" for value in CARD_VALUES for suit in SUITS]

#Q-Table
Q = defaultdict(lambda: np.zeros(3)) #0 = Hit, 1 = Stay, 2 = Double-down

########################### USEFUL FUNCTIONS #############################
#Function that checks if we have a natural Blackjack (first 2 cards)
def check_natural_blackjack(hand):
    return len(hand) == 2 and calculate_hand_value(hand) == 21

#Function that calculates the hand sum
def calculate_hand_value(hand):
    total = 0
    aces = 0
    for card in hand:
        value = card.split()[0]
        if value == "A":
            aces += 1
        else:
            total += CARD_VALUES[value]
    for _ in range(aces):
        total += 11 if total + 11 <= 21 else 1
    return total

#Function that calculates the dealer hand sum
def calculate_dealer_hand_value(hand):
    total = 0
    aces = 0
    for card in hand:
        value = card.split()[0]
        if value == "A":
            aces += 1
        else:
            total += CARD_VALUES[value]
    for _ in range(aces):
        total += 11 if 17 <= total + 11 <= 21 else 1 #If the dealer's hand is between 17 and 21 with an Ace turn to 11 it must stay this way
    return total

########################### BASIC BLACKJACK FUNCTIONS ####################
#Function that returns the state
def get_state_basic(player_hand, dealer_card):
    total = calculate_hand_value(player_hand)
    usable_ace = 1 if any(card.startswith("A") and calculate_hand_value(player_hand) <= 21 for card in player_hand) else 0
    value = dealer_card.split()[0]
    dealer_value = 11 if value == "A" else CARD_VALUES[value]
    return (total, dealer_value, usable_ace)

#Function that chooses the action to be performed, while handling the case where double-down is not allowed
def choose_action(state, epsilon, player_hand):
    valid_actions = [0, 1]  #hit, stay
    if len(player_hand) == 2:
        valid_actions.append(2)  #allow double-down only on 2 cards

    if np.random.rand() < epsilon: #explore
        return np.random.choice(valid_actions)

    #exploit: Choose the best valid action only among allowed ones
    q_values = Q[state]
    best_action = max(valid_actions, key=lambda a: q_values[a])
    return best_action

#Function that handles the dealer's turn
def play_dealer_basic(deck, dealer_hand):
    while calculate_dealer_hand_value(dealer_hand) < 17:
        dealer_hand.append(deck.pop())
    return calculate_dealer_hand_value(dealer_hand)

#Q-Learning Training basic
def Q_learning_basic():
  EPISODES = 2_000_000
  alpha = 0.05 #Constant learning rate
  gamma = 1 #Discount isn't needed
  epsilon = 1.0
  min_epsilon = 0.005 #E-greedy with epsilon starting from 1 and linearly decreasing to min_epsilon during the entire training phase
  decay = 0.999995 #rate with which exploration is decreased

  for episode in range(EPISODES):
      deck = DECK.copy()
      random.shuffle(deck)

      player_hand = [deck.pop(), deck.pop()]
      dealer_hand = [deck.pop(), deck.pop()]
      dealer_card = dealer_hand[1]

      player_bet = 1 #player can only bet 1 euro for the tabular case

      state = get_state_basic(player_hand, dealer_card)
      done = False

      if check_natural_blackjack(player_hand):
            if check_natural_blackjack(dealer_hand):
                reward = 0
                Q[state][1] += alpha * (reward - Q[state][1])
                continue
            reward = player_bet * 1.5 #in natural blackjack, the player gets 1.5x his initial bet and the dealer doesn't get to play
            Q[state][1] += alpha * (reward - Q[state][1]) #when the player hits a natural blackjack, we count it as a stay automatically (could have put it inside the while to check it using our policy)
            continue

      while not done:
          action = choose_action(state, epsilon, player_hand)
          if action == 2: #double-down action
              player_bet = player_bet * 2 #double the player's bet
              player_hand.append(deck.pop()) #only one card can be drawn after doubling-down
              player_total = calculate_hand_value(player_hand)
              if player_total > 21:
                  reward = -player_bet
              else:
                  dealer_total = play_dealer_basic(deck, dealer_hand)
                  if dealer_total > 21 or player_total > dealer_total:
                      reward = player_bet
                  elif player_total < dealer_total:
                      reward = -player_bet
                  else:
                      reward = 0
              done = True
              new_state = None
          elif action == 0: #hit action
              player_hand.append(deck.pop())
              new_state = get_state_basic(player_hand, dealer_card)
              player_total = calculate_hand_value(player_hand)
              if player_total > 21:
                  reward = -1
                  done = True
              else:
                  reward = 0
          else: #stay action
              done = True
              player_total = calculate_hand_value(player_hand)
              dealer_total = play_dealer_basic(deck, dealer_hand)
              if dealer_total > 21 or player_total > dealer_total:
                  reward = 1
              elif player_total < dealer_total:
                  reward = -1
              else:
                  reward = 0
              new_state = None

          if not done: #Q-learning formula from the literature for non-terminal states
              Q[state][action] += alpha * (reward + gamma * np.max(Q[new_state]) - Q[state][action])
              state = new_state
          else: #Q-learning formula from the literature for terminal states
              Q[state][action] += alpha * (reward - Q[state][action])

      #if episode % 10000 == 0 and episode > 0:
          #print(f"Episode {episode} completed - Epsilon: {epsilon:.4f}")

      epsilon = max(min_epsilon, epsilon * decay) #minimum exploration is 0.5% (practically 0)

  return Q

#Evaluation of learned policy of basic
def evaluate_policy_basic(episodes=100000):
    wins = draws = losses = 0
    balance = 0
    total_bet = 0
    for episode in range(episodes):
        initial_bet = 1
        total_bet += initial_bet
        deck = DECK.copy()
        random.shuffle(deck)
        player_hand = [deck.pop(), deck.pop()]
        dealer_hand = [deck.pop(), deck.pop()]
        dealer_card = dealer_hand[1]
        state = get_state_basic(player_hand, dealer_card)

        if check_natural_blackjack(player_hand):
            if check_natural_blackjack(dealer_hand):
                balance += initial_bet
                draws += 1
                continue
            balance += 1.5*initial_bet + initial_bet
            wins += 1
            continue

        while True:
            if len(player_hand) == 2:
                valid_actions = [0, 1, 2] #double-down is allowed
            else:
                valid_actions = [0, 1] #double-down is not allowed

            q_values = Q[state] #choose the best action among valid ones
            action = max(valid_actions, key=lambda a: q_values[a])
            if action == 2: #double-down action
                total_bet += initial_bet
                initial_bet = 2*initial_bet
                player_hand.append(deck.pop()) #only one card can be drawn after doubling-down
                if calculate_hand_value(player_hand) > 21:
                    losses += 1
                break
            elif action == 0: #hit action
                new_card = deck.pop()
                player_hand.append(new_card)
                state = get_state_basic(player_hand, dealer_card)
                if calculate_hand_value(player_hand) > 21:
                    losses += 1
                    break
            else: #stay action
                break

        if calculate_hand_value(player_hand) <= 21:
            player_total = calculate_hand_value(player_hand)
            dealer_total = play_dealer_basic(deck, dealer_hand)
            if dealer_total > 21 or player_total > dealer_total:
                balance += 2*initial_bet
                wins += 1
            elif player_total < dealer_total:
                losses += 1
            else:
                balance += initial_bet
                draws += 1

    print(f"Out of {wins + losses + draws} games:")
    print(f"Wins:  {wins / episodes:.2%}")
    print(f"Draws: {draws / episodes:.2%}")
    print(f"Losses:{losses / episodes:.2%}")
    print(f"Balance: {int(balance)}")
    print(f"Total bet: {int(total_bet)}")
    print(f"Profit win ratio: {50 - (1 - balance / total_bet)*100:.2f}%")

#Plot policy function for basic
def plot_policy_basic(Q):
    def get_policy(usable_ace):
        policy = np.zeros((10, 10))  # player 12–21, dealer 2–11
        for player_total in range(12, 22):
            for dealer_card in range(2, 12):
                state = (player_total, dealer_card, int(usable_ace))
                if state in Q:
                    action = np.argmax(Q[state])
                else:
                    action = 0  # default to hit
                policy[player_total - 12, dealer_card - 2] = action
        return policy

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    action_labels = ['Hit', 'Stay', 'Double']
    cmap = sns.color_palette("coolwarm", 3)  # or choose any 3-color palette

    for i, usable_ace in enumerate([False, True]):
        policy = get_policy(usable_ace)
        ax = axes[i]
        sns.heatmap(policy,
                    cmap=cmap,
                    xticklabels=range(2, 12),
                    yticklabels=range(12, 22),
                    cbar=False,
                    ax=ax,
                    linewidths=.5,
                    linecolor='gray',
                    square=True,
                    vmin=0, vmax=2)  # Ensure 3-level scale: 0, 1, 2
        ax.set_title(f'Policy (Usable Ace: {usable_ace})')
        ax.set_xlabel('Dealer Showing')
        ax.set_ylabel('Player Total')
        ax.invert_yaxis()

    # Custom legend
    legend_patches = [
        mpatches.Patch(color=cmap[0], label='Hit'),
        mpatches.Patch(color=cmap[1], label='Stay'),
        mpatches.Patch(color=cmap[2], label='Double-down'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize='large')

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()
##########################################################################

########################### CARD COUNTING BLACKJACK FUNCTIONS ####################
#Function that returns the count state (1 -> High, 0 -> Neutral, -1 -> Low)
def calculate_count_state(count):
    if count > 3:
      return 1
    elif count < -3:
      return -1
    else:
      return 0

#Function that calculates the initial hands' count
def calculate_count(player_hand, dealer_card, count):
    player_card_values = [CARD_VALUES[card.split()[0]] for card in player_hand]
    for i in range(len(player_card_values)):
      if 2 <= player_card_values[i] <= 6:
        count += 1
      elif 10 <= player_card_values[i] <= 11:
        count -= 1

    dealer_card_value = CARD_VALUES[dealer_card.split()[0]]
    if 2 <= dealer_card_value <= 6:
        count += 1
    elif 10 <= dealer_card_value <= 11:
        count -= 1

    return count

#Function that adjusts the count based on a new card
def adjust_count(new_card, count):
    new_card_value = CARD_VALUES[new_card.split()[0]] #only one card can be drawn after doubling-down
    if 2 <= new_card_value <= 6:
        count += 1
    elif 10 <= new_card_value <= 11:
        count -= 1

    return count

#Function that returns the state
def get_state_card_counting(player_hand, dealer_card, count):
    total = calculate_hand_value(player_hand)
    usable_ace = 1 if any(card.startswith("A") and calculate_hand_value(player_hand) <= 21 for card in player_hand) else 0
    value = dealer_card.split()[0]
    dealer_value = 11 if value == "A" else CARD_VALUES[value]
    count_state = calculate_count_state(count)
    return (total, dealer_value, usable_ace, count_state)

#Function that handles the dealer's turn
def play_dealer_card_counting(deck, dealer_hand, count):
    while calculate_dealer_hand_value(dealer_hand) < 17:
        if len(deck) == 0: #Every time the deck is empty, we get a new shuffled deck and reset the card count
            deck = DECK.copy()
            random.shuffle(deck)
            count = 0

        dealer_new_card = deck.pop()
        dealer_hand.append(dealer_new_card)

        dealer_new_card_value = CARD_VALUES[dealer_new_card.split()[0]]
        if 2 <= dealer_new_card_value <= 6:
            count += 1
        elif 10 <= dealer_new_card_value <= 11:
            count -= 1
    return calculate_dealer_hand_value(dealer_hand), count, deck

#Q-Learning Training for card counting
def Q_learning_card_counting():
  EPISODES = 6_000_000 #number of episodes is tripled because the state space is tripled
  alpha = 0.05
  gamma = 1
  epsilon = 1.0
  min_epsilon = 0.005
  decay = 0.999999

  deck = DECK.copy()
  random.shuffle(deck)
  count = 0

  for episode in range(EPISODES):
      if len(deck) <= 10: #Every time we have 10 or less cards we re-shuffle
        deck = DECK.copy()
        random.shuffle(deck)
        count = 0

      player_hand = [deck.pop(), deck.pop()]
      dealer_hand = [deck.pop(), deck.pop()]
      dealer_card = dealer_hand[1]

      player_bet = 1

      count = calculate_count(player_hand, dealer_card, count)

      state = get_state_card_counting(player_hand, dealer_card, count)
      done = False

      if check_natural_blackjack(player_hand):
            if check_natural_blackjack(dealer_hand):
                count = adjust_count(dealer_hand[0], count)
                reward = 0
                Q[state][1] += alpha * (reward - Q[state][1])
                continue
            reward = player_bet * 1.5 #in natural blackjack, the player gets 1.5x his initial bet and the dealer doesn't get to play
            Q[state][1] += alpha * (reward - Q[state][1]) #when the player hits a natural blackjack, we count it as a stay automatically (could have put it inside the while to check it using our policy)
            continue

      while not done:
          action = choose_action(state, epsilon, player_hand)
          if action == 2: #double-down action
              player_bet = player_bet * 2 #double the player's bet
              player_new_card = deck.pop() #in double-down we only get 1 more card
              player_hand.append(player_new_card)
              count = adjust_count(player_new_card, count)

              player_total = calculate_hand_value(player_hand)
              if player_total > 21:
                  reward = -player_bet
              else:
                  dealer_total, count, deck = play_dealer_card_counting(deck, dealer_hand, count)
                  if dealer_total > 21 or player_total > dealer_total:
                      reward = player_bet
                  elif player_total < dealer_total:
                      reward = -player_bet
                  else:
                      reward = 0
              done = True
              new_state = None
          elif action == 0: #hit action
              player_new_card = deck.pop()
              player_hand.append(player_new_card)
              count = adjust_count(player_new_card, count)

              new_state = get_state_card_counting(player_hand, dealer_card, count)
              player_total = calculate_hand_value(player_hand)
              if player_total > 21:
                  reward = -1
                  done = True
              else:
                  reward = 0
          else: #stay action
              done = True
              player_total = calculate_hand_value(player_hand)
              dealer_total, count, deck = play_dealer_card_counting(deck, dealer_hand, count)
              if dealer_total > 21 or player_total > dealer_total:
                  reward = 1
              elif player_total < dealer_total:
                  reward = -1
              else:
                  reward = 0
              new_state = None

          if not done:
              Q[state][action] += alpha * (reward + gamma * np.max(Q[new_state]) - Q[state][action])
              state = new_state
          else:
              Q[state][action] += alpha * (reward - Q[state][action])

      #if episode % 10000 == 0 and episode > 0:
          #print(f"Episode {episode} completed - Epsilon: {epsilon:.4f}")

      epsilon = max(min_epsilon, epsilon * decay)

  return Q

#Evaluation of learned policy of card counting
def evaluate_policy_card_counting(episodes=100000):
    deck = DECK.copy()
    random.shuffle(deck)
    count = 0
    wins = draws = losses = 0
    balance = 0
    total_bet = 0
    for _ in range(episodes):
        initial_bet = 1
        total_bet += initial_bet
        if len(deck) <= 10:
            deck = DECK.copy()
            random.shuffle(deck)
            count = 0
        player_hand = [deck.pop(), deck.pop()]
        dealer_hand = [deck.pop(), deck.pop()]
        dealer_card = dealer_hand[1]
        count = calculate_count(player_hand, dealer_card, count)

        state = get_state_card_counting(player_hand, dealer_card, count)

        if check_natural_blackjack(player_hand):
            if check_natural_blackjack(dealer_hand):
                count = adjust_count(dealer_hand[0], count)
                balance += initial_bet
                draws += 1
                continue
            balance += 1.5*initial_bet + initial_bet
            wins += 1
            continue

        while True:
            if len(player_hand) == 2:
                valid_actions = [0, 1, 2] #double-down is allowed
            else:
                valid_actions = [0, 1] #double-down is not allowed

            q_values = Q[state] #choose the best action among valid ones
            action = max(valid_actions, key=lambda a: q_values[a])
            if action == 2: #double-down action
                total_bet += initial_bet
                initial_bet = 2*initial_bet
                player_new_card = deck.pop() #in double-down we only get 1 more card1

                player_hand.append(player_new_card)
                count = adjust_count(player_new_card, count)

                state = get_state_card_counting(player_hand, dealer_card, count)
                if calculate_hand_value(player_hand) > 21:
                    losses += 1
                break
            elif action == 0:  #hit action
                player_new_card = deck.pop()
                player_hand.append(player_new_card)
                count = adjust_count(player_new_card, count)

                state = get_state_card_counting(player_hand, dealer_card, count)
                if calculate_hand_value(player_hand) > 21:
                    losses += 1
                    break
            else:  #stay action
                break

        if calculate_hand_value(player_hand) <= 21:
            player_total = calculate_hand_value(player_hand)
            dealer_total, count, deck = play_dealer_card_counting(deck, dealer_hand, count)
            if dealer_total > 21 or player_total > dealer_total:
                balance += 2*initial_bet
                wins += 1
            elif player_total < dealer_total:
                losses += 1
            else:
                balance += initial_bet
                draws += 1

    print(f"Out of {wins + losses + draws} games:")
    print(f"Wins:  {wins / episodes:.2%}")
    print(f"Draws: {draws / episodes:.2%}")
    print(f"Losses:{losses / episodes:.2%}")
    print(f"Balance: {int(balance)}")
    print(f"Total bet: {int(total_bet)}")
    print(f"Profit win ratio: {50 - (1 - balance / total_bet)*100:.2f}%")

#Plot policy function for card counting
def plot_policy_card_counting(Q):
    def get_policy(usable_ace, count_state):
        policy = np.zeros((10, 10))  #player total 12-21, dealer 2-11
        for player_total in range(12, 22):
            for dealer_card in range(2, 12):
                state = (player_total, dealer_card, int(usable_ace), count_state)
                if state in Q:
                    action = np.argmax(Q[state])
                else:
                    action = 0  #default to hit
                policy[player_total - 12, dealer_card - 2] = action
        return policy

    fig, axes = plt.subplots(3, 2, figsize=(14, 18))

    #color setup for 3 actions
    action_labels = ['Hit', 'Stay', 'Double-down']
    cmap = sns.color_palette("coolwarm", 3)  #use a 3-color scale
    count_labels = ['Low Count (< -3)', 'Neutral Count (-3 to 3)', 'High Count (> 3)']

    for ci, count_state in enumerate([-1, 0, 1]):
        for i, usable_ace in enumerate([False, True]):
            policy = get_policy(usable_ace, count_state)
            ax = axes[ci, i]
            sns.heatmap(policy,
                        cmap=cmap,
                        xticklabels=range(2, 12),
                        yticklabels=range(12, 22),
                        cbar=False,
                        ax=ax,
                        linewidths=.5,
                        linecolor='gray',
                        square=True,
                        vmin=0, vmax=2)  # 0=Hit, 1=Stay, 2=Double-down
            ax.set_title(f'Policy (Usable Ace: {usable_ace}, {count_labels[ci]})')
            ax.set_xlabel('Dealer Showing')
            ax.set_ylabel('Player Total')
            ax.invert_yaxis()

    #custom legend for 3 actions
    legend_patches = [
        mpatches.Patch(color=cmap[0], label='Hit'),
        mpatches.Patch(color=cmap[1], label='Stay'),
        mpatches.Patch(color=cmap[2], label='Double-down'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize='large')

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()
##########################################################################

def main_menu():
    while True:
        print("\n" + "="*40)
        print("Blackjack Q-Learning Console Menu")
        print("="*40)
        print("1. Train Agent for basic Blackjack")
        print("2. Train Agent for card counting Blackjack")
        print("3. Exit")
        print("="*40)

        choice = input("Select an option (1-3): ").strip()

        if choice == '1': #Train for basic Blackjack
            print("\nTraining the agent for basic Blackjack... This might take a while.")
            Q_learning_basic()
            evaluate_policy_basic()
            plot_policy_basic(Q)
            time.sleep(1)
        elif choice == '2': #Train for card counting Blackjack
            print("\nTraining the agent for card counting Blackjack... This might take a while.")
            Q_learning_card_counting()
            evaluate_policy_card_counting()
            plot_policy_card_counting(Q)
            time.sleep(2)
        elif choice == '3': #Exit
            print("\nThanks for playing! Goodbye")
            time.sleep(1)
            sys.exit()
        else:
            print("\nInvalid input. Please enter a number from 1 to 3.")
            time.sleep(1)


if __name__ == "__main__":
    main_menu()


Blackjack Q-Learning Console Menu
1. Train Agent for basic Blackjack
2. Train Agent for card counting Blackjack
3. Exit
